In [1]:
import pandas as pd
import numpy as np

In [2]:

df_train = pd.read_csv("../data/raw/train.csv") # Los '..' backean atrás de la carpeta
df_hevents = pd.read_csv("../data/raw/holidays_events.csv")
df_oil = pd.read_csv("../data/raw/oil.csv")
df_sample = pd.read_csv("../data/raw/sample_submission.csv")
df_stores = pd.read_csv("../data/raw/stores.csv")
df_test = pd.read_csv("../data/raw/test.csv")
df_transactions = pd.read_csv("../data/raw/transactions.csv")


In [3]:
dataframes = {
    "df_train": df_train,
    "df_hevents": df_hevents,
    "df_oil": df_oil,
    "df_sample": df_sample,
    "df_stores": df_stores,
    "df_transactions": df_transactions
}
for name, df in dataframes.items():
    print(f'\nExploración de: {name} \n')
    print('DTYPES:\n',df.dtypes)
    print('\nSHAPE: ',df.shape)
    print('\nDuplicated: ',df.duplicated().sum())
    print('\nNulls:\n',df.isnull().sum())
    print('\nDATAFRAME:\n',df.head(3),'\n ------ END ------')
    


Exploración de: df_train 

DTYPES:
 id               int64
date               str
store_nbr        int64
family             str
sales          float64
onpromotion      int64
dtype: object

SHAPE:  (3000888, 6)

Duplicated:  0

Nulls:
 id             0
date           0
store_nbr      0
family         0
sales          0
onpromotion    0
dtype: int64

DATAFRAME:
    id        date  store_nbr      family  sales  onpromotion
0   0  2013-01-01          1  AUTOMOTIVE    0.0            0
1   1  2013-01-01          1   BABY CARE    0.0            0
2   2  2013-01-01          1      BEAUTY    0.0            0 
 ------ END ------

Exploración de: df_hevents 

DTYPES:
 date            str
type            str
locale          str
locale_name     str
description     str
transferred    bool
dtype: object

SHAPE:  (350, 6)

Duplicated:  0

Nulls:
 date           0
type           0
locale         0
locale_name    0
description    0
transferred    0
dtype: int64

DATAFRAME:
          date     type    lo

### df_train

#### PK

In [4]:

df_train.drop_duplicates(subset=['date','store_nbr','family']).shape[0] == len(df_train)


True

#### cleaning dtypes

In [5]:

df_train['date'] = pd.to_datetime(df_train['date'])
df_train.dtypes


id                      int64
date           datetime64[us]
store_nbr               int64
family                    str
sales                 float64
onpromotion             int64
dtype: object

#### Date range

In [6]:

df_train['date'].min(), df_train['date'].max()


(Timestamp('2013-01-01 00:00:00'), Timestamp('2017-08-15 00:00:00'))

#### Avaiable dates

In [7]:
calendar = pd.date_range(df_train['date'].min(), df_train['date'].max())
print(len(calendar), df_train['date'].nunique())
print(calendar.difference(df_train['date'].unique()))

## Closed on Christmas
## Binary work day -- Bridge and word kays

1688 1684
DatetimeIndex(['2013-12-25', '2014-12-25', '2015-12-25', '2016-12-25'], dtype='datetime64[us]', freq=None)


#### Family x Stores

In [8]:
n_stores = df_train['store_nbr'].nunique()
n_family = df_train['family'].nunique()
n_stores * n_family

1782

In [9]:
# Confirmation

df_train.groupby(['store_nbr','family']).ngroups

1782

### df_transactions

#### PK

In [10]:

df_transactions.drop_duplicates(subset=['date','store_nbr']).shape[0] == len(df_transactions) ## PK


True

#### Cleaning dtypes

In [11]:
df_transactions['date'] = pd.to_datetime(df_train['date'])
df_transactions.dtypes

date            datetime64[us]
store_nbr                int64
transactions             int64
dtype: object

#### PK

In [12]:

df_transactions.drop_duplicates(subset=['date','store_nbr']).shape[0] == len(df_transactions) ## PK


False

#### Stores opened after the first date

In [13]:
first = df_transactions.groupby('store_nbr')['date'].min()
first.value_counts()

date
2013-01-01    46
2013-01-21     1
2013-01-25     1
2013-01-27     1
2013-01-22     1
2013-01-04     1
2013-01-26     1
2013-02-13     1
2013-01-14     1
Name: count, dtype: int64

### df_hevents

#### PK

In [14]:
df_hevents.drop_duplicates(subset=['date','description']).shape[0] == len(df_hevents) ## PK

True

### df_stores

#### PK

In [15]:
df_stores.drop_duplicates(subset=['store_nbr']).shape[0] == len(df_stores) ## PK

True

#### Sobran los national

In [16]:
locales = set(df_hevents['locale_name'].unique())
ciudades = set(df_stores['city'].unique())
estados = set(df_stores['state'].unique())

locales - ciudades - estados

{'Ecuador'}

### df_oil

#### Cleaning data types

In [17]:
df_oil['date'] = pd.to_datetime(df_oil['date'])

In [18]:
df_oil.isnull().sum()

date           0
dcoilwtico    43
dtype: int64

#### PK

In [19]:
df_oil.drop_duplicates(subset=['date']).shape[0] == len(df_oil) ## PK

True

In [20]:
calendar = pd.date_range(df_train['date'].min(), df_train['date'].max())
calendar.difference(df_oil['date'])

DatetimeIndex(['2013-01-05', '2013-01-06', '2013-01-12', '2013-01-13',
               '2013-01-19', '2013-01-20', '2013-01-26', '2013-01-27',
               '2013-02-02', '2013-02-03',
               ...
               '2017-07-15', '2017-07-16', '2017-07-22', '2017-07-23',
               '2017-07-29', '2017-07-30', '2017-08-05', '2017-08-06',
               '2017-08-12', '2017-08-13'],
              dtype='datetime64[us]', length=482, freq=None)

### df_test

In [21]:
df_test = pd.read_csv("../data/raw/test.csv")
df_test['date'] = pd.to_datetime(df_test['date'])
df_test['date'].min(), df_test['date'].max()

(Timestamp('2017-08-16 00:00:00'), Timestamp('2017-08-31 00:00:00'))

### Buscando ceros

In [22]:
df_train['es_cero'] = df_train['sales'] == 0
ceros_por_serie = df_train.groupby(['store_nbr','family'])['es_cero'].mean()

In [23]:
ceros_por_serie.describe()

count    1782.000000
mean        0.312951
std         0.322887
min         0.002969
25%         0.002969
50%         0.306413
75%         0.504008
max         1.000000
Name: es_cero, dtype: float64

In [24]:
(ceros_por_serie > 0.9).sum()

np.int64(173)

In [25]:
series_muertas = ceros_por_serie[ceros_por_serie > 0.9]
series_muertas.index.get_level_values('family').value_counts()

family
BOOKS                         52
BABY CARE                     43
LAWN AND GARDEN               16
SCHOOL AND OFFICE SUPPLIES    14
LADIESWEAR                    14
HOME APPLIANCES                4
PET SUPPLIES                   3
MAGAZINES                      2
AUTOMOTIVE                     1
BEAUTY                         1
BEVERAGES                      1
BREAD/BAKERY                   1
CELEBRATION                    1
CLEANING                       1
DAIRY                          1
DELI                           1
EGGS                           1
FROZEN FOODS                   1
GROCERY I                      1
GROCERY II                     1
HARDWARE                       1
HOME AND KITCHEN I             1
HOME AND KITCHEN II            1
HOME CARE                      1
LINGERIE                       1
LIQUOR,WINE,BEER               1
MEATS                          1
PERSONAL CARE                  1
PLAYERS AND ELECTRONICS        1
POULTRY                        1
PRE